# בניית וקטורים, שש חבילות הפצה ופרסום ב־GitHub

המחברת מורידה את מסד הנתונים והאינדקס המילולי לפי DB_RELEASE_TAG, מייצרת אינדקס סמנטי בשני T4, מורידה PDF מאוצריא ומפרסמת שש חבילות. כל חבילה נארזת, עולה לטיוטה ונמחקת מהדיסק לפני החבילה הבאה. החבילה המלאה seforim_bundle היא ברירת המחדל.

הגדירו GPU T4 x2, הפעילו Internet, והוסיפו HF_TOKEN עם גישה למודל ו־GH_TOKEN עם Contents: read and write למאגרים arieldaniely/SeforimLibrary ו־arieldaniely/Zayit. סמנו את הסודות כזמינים למחברת. אין להדביק טוקנים בקוד.


הזינו את תג המסד אחרי סיום GitHub Actions. כל שש החבילות נדחסות ברמת Zstandard 22. הפרסום משתמש בתג ייחודי חדש <DB_RELEASE_TAG>-full-<מזהה>, גם אם כבר קיימת גרסת FULL לאותו מסד, ומשאיר את גרסת הבסיס ללא שינוי. בסוף המחברת מפרסמת אוטומטית. כלי ההפצה כלולים במחברת ואין צורך לדחוף את שינוייהם לגיטהאב לפני ההרצה. קוד מנוע האינדוקס נמשך מהמאגר; בחרו ZAYIT_REF שמכיל תמיכת int8. הזמן והדיסק של הריצה המלאה טרם נמדדו; במקרה כישלון ההפצות נשארות בטיוטה.

שני GPU מייצרים קובצי ביניים; המחברת בונה מהם אינדקס Lucene אחד מאוחד (shardCount=1), עם מקטע אחד ווקטורים מנורמלים שכומתו ל־int8. קוד הבנייה המאוחד כלול בתא release-tools. יש להריץ מחדש ולהתקין את החבילה החדשה.


In [ ]:
%pip install -q zstandard huggingface_hub safetensors onnxruntime 'transformers<5' requests regex


In [ ]:
import torch
import shutil
from pathlib import Path

assert torch.cuda.device_count() == 2, 'בחרו GPU T4 x2 בהגדרות המחברת'
print([torch.cuda.get_device_name(i) for i in range(2)])
scratch_dir = Path('/kaggle/temp')
scratch_dir.mkdir(parents=True, exist_ok=True)
print('Free scratch space (GiB):', round(shutil.disk_usage(scratch_dir).free / 2**30, 1))


In [ ]:
import subprocess
import os

# Set ZAYIT_REF to the branch containing the int8 changes after publishing.
code_ref = os.environ.get('ZAYIT_REF', 'master')

repo = Path('/kaggle/temp/Zayit')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', code_ref,
                    'https://github.com/arieldaniely/Zayit.git', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'submodule', 'update', '--init', 'SeforimLibrary'], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', code_ref], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', code_ref], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
    subprocess.run(['git', '-C', str(repo), 'submodule', 'update', '--init', 'SeforimLibrary'], check=True)
assert (repo / 'SeforimLibrary/search/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/search/BuildSemanticIndexFromVectors.kt').is_file()
print('Code:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())
assert (repo / 'scripts/kaggle_embed_vectors.py').is_file()
assert (repo / 'SeforimLibrary/search/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/search/Int8Vectors.kt').is_file(), 'יש לבחור ענף שמכיל את שינויי int8'


In [ ]:
# Release tooling is embedded and kept in sync with the repository.
embedded_files = {'scripts/kaggle_semantic_bundle.py': '"""End-to-end Kaggle T4 x2 build from the published SeforimLibrary DB release."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport sqlite3\nimport subprocess\nimport sys\nimport tarfile\nfrom pathlib import Path\n\nimport requests\nimport torch\nimport zstandard\nfrom huggingface_hub import hf_hub_download, snapshot_download\n\n\nSOURCE_REPO = "ArieLLL123/judaic-semantic-teacher-8x512-retrieval-v5-round2"\nSOURCE_REVISION = "8d7016cf472fb436dbd3ac843ef4d407c28d08f4"\nONNX_REPO = "ArieLLL123/Meivin-Embed-0.1"\nMODEL_SHA = "659226865abd3a1bc833565ae6b2e2f48abdd7136285824a12966d4d3294cbf8"\nTOKENIZER_SHA = "0664287976ecb078bdfd8f5e5515dc87d8cb7f985a79a481aa1cdf7a7321c0e9"\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for chunk in iter(lambda: source.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef download(url: str, target: Path, expected_sha: str, *, token: str | None = None) -> None:\n    if target.is_file() and sha256(target) == expected_sha:\n        print(f"Verified existing {target.name}", flush=True)\n        return\n    partial = target.with_name(target.name + ".download")\n    offset = partial.stat().st_size if partial.exists() else 0\n    headers = {"Range": f"bytes={offset}-"} if offset else {}\n    if token:\n        headers.update(Authorization=f"Bearer {token}", Accept="application/octet-stream")\n    with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:\n        response.raise_for_status()\n        if offset and response.status_code != 206:\n            offset = 0\n        with partial.open("ab" if offset else "wb") as output:\n            for chunk in response.iter_content(chunk_size=1 << 20):\n                if chunk:\n                    output.write(chunk)\n    actual = sha256(partial)\n    if actual != expected_sha:\n        raise ValueError(f"SHA-256 mismatch for {target.name}: {actual}")\n    partial.replace(target)\n    print(f"Downloaded and verified {target.name}", flush=True)\n\n\ndef release_assets(repository: str, tag: str, token: str | None) -> list[dict]:\n    from urllib.parse import quote\n\n    headers = {"Accept": "application/vnd.github+json"}\n    if token:\n        headers["Authorization"] = f"Bearer {token}"\n    response = requests.get(\n        f"https://api.github.com/repos/{repository}/releases/tags/{quote(tag, safe=\'\')}",\n        headers=headers, timeout=(30, 120),\n    )\n    response.raise_for_status()\n    assets = response.json()["assets"]\n    for prefix in ("seforim_bundle-database-only.tar.zst", "seforim_bundle.tar.zst"):\n        single = [asset for asset in assets if asset["name"] == prefix]\n        parts = sorted(\n            (asset for asset in assets if re.fullmatch(re.escape(prefix) + r"\\.part\\d+", asset["name"])),\n            key=lambda asset: int(asset["name"].rsplit("part", 1)[1]),\n        )\n        selected = parts or single\n        if not selected:\n            continue\n        if parts and [int(asset["name"].rsplit("part", 1)[1]) for asset in parts] != list(range(1, len(parts) + 1)):\n            raise ValueError("Database archive parts are not consecutive")\n        for asset in selected:\n            if not re.fullmatch(r"sha256:[0-9a-f]{64}", asset.get("digest") or ""):\n                raise ValueError(f"Release asset has no SHA-256 digest: {asset[\'name\']}")\n        return selected\n    raise FileNotFoundError(f"No database bundle in {repository}@{tag}")\n\n\nclass MultipartStream:\n    """Read split .zst files as one compressed byte stream."""\n\n    def __init__(self, parts: list[Path]) -> None:\n        self.parts = iter(parts)\n        self.current = next(self.parts).open("rb")\n\n    def read(self, size: int = -1) -> bytes:\n        if size < 0:\n            return b"".join(iter(lambda: self.read(1 << 20), b""))\n        while True:\n            data = self.current.read(size)\n            if data:\n                return data\n            self.current.close()\n            next_part = next(self.parts, None)\n            if next_part is None:\n                return b""\n            self.current = next_part.open("rb")\n\n    def close(self) -> None:\n        self.current.close()\n\n\ndef extract_database(parts: list[Path], destination: Path) -> None:\n    if destination.is_file():\n        print(f"Using existing {destination}", flush=True)\n        return\n    stream = MultipartStream(parts)\n    temporary = destination.with_suffix(".building")\n    try:\n        with contextlib.closing(stream), zstandard.ZstdDecompressor().stream_reader(stream) as reader:\n            with tarfile.open(fileobj=reader, mode="r|") as archive:\n                for member in archive:\n                    if member.isfile() and Path(member.name).name == "seforim.db":\n                        with archive.extractfile(member) as source, temporary.open("wb") as output:\n                            shutil.copyfileobj(source, output, 1 << 20)\n                        temporary.replace(destination)\n                        print(f"Extracted {destination} ({destination.stat().st_size:,} bytes)", flush=True)\n                        return\n        raise FileNotFoundError("seforim.db was not found in the release archive")\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef get_hf_token() -> str:\n    token = os.getenv("HF_TOKEN")\n    if not token:\n        from kaggle_secrets import UserSecretsClient\n\n        token = UserSecretsClient().get_secret("HF_TOKEN")\n    if not token:\n        raise RuntimeError("Set an approved HF_TOKEN in Kaggle Secrets")\n    return token\n\n\ndef fetch_models(work: Path) -> tuple[Path, Path]:\n    token = get_hf_token()\n    source = Path(snapshot_download(\n        repo_id=SOURCE_REPO, revision=SOURCE_REVISION, token=token,\n        allow_patterns=["round2/final/*", "round2/final/**/*"],\n        local_dir=work / "checkpoint-download",\n    )) / "round2" / "final"\n    for name in ("sentence_encoder_config.json", "projection.safetensors", "backbone/model.safetensors",\n                 "backbone/config.json", "tokenizer/tokenizer.json"):\n        if not (source / name).is_file():\n            raise FileNotFoundError(source / name)\n    model_dir = work / "model"\n    model_dir.mkdir(exist_ok=True)\n    for name, expected in (("seforim-embed-round2-int8.onnx", MODEL_SHA),\n                           ("tokenizer.json", TOKENIZER_SHA)):\n        path = Path(hf_hub_download(repo_id=ONNX_REPO, filename=name, token=token))\n        if sha256(path) != expected:\n            raise ValueError(f"Unexpected {name} from Hugging Face")\n        shutil.copy2(path, model_dir / name)\n    return source, model_dir\n\n\ndef install_java25(work: Path) -> Path:\n    java_home = os.getenv("JAVA_HOME")\n    if java_home:\n        result = subprocess.run([str(Path(java_home) / "bin/java"), "-version"], capture_output=True, text=True)\n        if \'version "25\' in result.stderr or \'version "25\' in result.stdout:\n            return Path(java_home)\n    archive = work / "jdk25.tar.gz"\n    if not archive.exists():\n        url = "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"\n        with requests.get(url, stream=True, timeout=(30, 120)) as response:\n            response.raise_for_status()\n            with archive.open("wb") as output:\n                for chunk in response.iter_content(1 << 20):\n                    if chunk:\n                        output.write(chunk)\n    directory = work / "jdk25"\n    directory.mkdir(exist_ok=True)\n    subprocess.run(["tar", "-xzf", str(archive), "-C", str(directory)], check=True)\n    homes = list(directory.glob("jdk-25*"))\n    if len(homes) != 1:\n        raise RuntimeError("Could not locate extracted JDK 25")\n    return homes[0]\n\n\ndef run_command(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:\n    print("Running:", " ".join(command[:4]), "...", flush=True)\n    subprocess.run(command, cwd=cwd, env=env, check=True)\n\n\ndef main(args: argparse.Namespace) -> None:\n    if torch.cuda.device_count() != 2 or not all("T4" in torch.cuda.get_device_name(i) for i in range(2)):\n        raise RuntimeError("Select Kaggle GPU T4 x2 in Notebook Settings")\n    args.work.mkdir(parents=True, exist_ok=True)\n    args.output.mkdir(parents=True, exist_ok=True)\n    token = os.getenv("GH_TOKEN")\n    if args.all_distributions and not token:\n        from kaggle_secrets import UserSecretsClient\n\n        token = UserSecretsClient().get_secret("GH_TOKEN")\n    if args.all_distributions:\n        from kaggle_publish_distributions import Publisher\n\n        final_tag = args.publish_tag or args.db_release_tag + \'-full\'\n        if final_tag == args.db_release_tag:\n            raise ValueError(\'Final release must use a new tag\')\n        publisher = Publisher(token)\n        args.publish_tag = publisher.allocate(args.db_repository, args.app_repository, final_tag, args.work)\n    assets = release_assets(args.db_repository, args.db_release_tag, token)\n    identity = {"repository": args.db_repository, "tag": args.db_release_tag,\n                "allDistributions": args.all_distributions,\n                "assets": [{"name": asset["name"], "digest": asset["digest"]} for asset in assets]}\n    marker = args.work / "database-source.json"\n    if marker.exists():\n        if json.loads(marker.read_text()) != identity:\n            raise RuntimeError("This work directory belongs to another database; select a new --work directory")\n    else:\n        if (args.work / "seforim.db").exists() or (args.work / "index-single").exists():\n            raise RuntimeError("Untracked previous build; select a new --work directory")\n        marker.write_text(json.dumps(identity), encoding="utf-8")\n    download_dir = args.work / "release"\n    download_dir.mkdir(exist_ok=True)\n    parts = []\n    for asset in assets:\n        name, digest = asset["name"], asset["digest"].removeprefix("sha256:")\n        path = download_dir / name\n        download(asset["url"] if token else asset["browser_download_url"], path, digest, token=token)\n        parts.append(path)\n    database = args.work / "seforim.db"\n    if args.all_distributions:\n        from kaggle_publish_distributions import extract_complete\n\n        extract_complete(parts, args.work, MultipartStream)\n        if not database.is_file():\n            raise FileNotFoundError(\'The source release does not contain seforim.db\')\n    else:\n        extract_database(parts, database)\n    database_sha = sha256(database)\n    print("Database SHA-256:", database_sha, flush=True)\n    with sqlite3.connect(f"file:{database}?mode=ro", uri=True) as connection:\n        print("Database rows:", connection.execute("SELECT COUNT(*) FROM line").fetchone()[0], flush=True)\n        if connection.execute("PRAGMA quick_check").fetchone()[0] != "ok":\n            raise ValueError("The extracted database failed SQLite quick_check")\n    for part in parts:\n        part.unlink()\n\n    checkpoint, model_dir = fetch_models(args.work)\n    java_home = install_java25(args.work)\n    env = dict(os.environ, JAVA_HOME=str(java_home), RAYON_NUM_THREADS="2", OMP_NUM_THREADS="2")\n    if args.all_distributions:\n        library = args.repo / "SeforimLibrary"\n        gradle = library / "gradlew"\n        gradle.chmod(gradle.stat().st_mode | 0o111)\n        tasks = []\n        if not all((database.parent / name).is_dir() and any((database.parent / name).iterdir())\n                   for name in (\'seforim.db.lucene\', \'seforim.db.lookup.lucene\')):\n            tasks.append(\':searchindex:buildLuceneIndexDefault\')\n        if not (database.parent / \'catalog.pb\').is_file():\n            tasks.append(\':catalog:buildCatalog\')\n        if not (database.parent / \'lexical.db\').is_file():\n            tasks.append(\':packaging:downloadLexicalDb\')\n        if not (database.parent / \'release_info.txt\').is_file():\n            tasks.append(\':packaging:writeReleaseInfo\')\n        if tasks:\n            run_command([str(gradle), *tasks, f\'-PseforimDb={database}\',\n                         f\'-PreleaseName={args.db_release_tag}\', \'-PinMemoryDb=false\',\n                         \'--no-daemon\', \'--no-configuration-cache\', \'--max-workers=4\'], cwd=library, env=env)\n    vectors_dir = args.work / "vectors-single"\n    vectors_dir.mkdir(exist_ok=True)\n    embed_script = args.repo / "scripts/kaggle_embed_vectors.py"\n    worker_files = [vectors_dir / f"worker-{gpu}.json" for gpu in range(2)]\n    vector_files = [vectors_dir / f"shard-{shard:02d}.bin" for shard in range(2)]\n    index = args.work / "index-single"\n    has_workers = all(f.is_file() and json.loads(f.read_text()).get("vectorEncoding") == "int8-maxabs-v1"\n                      for f in worker_files)\n    has_vectors = all(f.is_file() for f in vector_files)\n    has_indexed = (index / "shard-00" / "semantic.properties").is_file()\n\n    if has_workers and (has_vectors or has_indexed):\n        print("GPU encoding already completed; skipping vector generation", flush=True)\n        counts = [json.loads(f.read_text()) for f in worker_files]\n        print("GPU workers:", counts, flush=True)\n    else:\n        processes = []\n        for gpu in range(2):\n            command = [\n                sys.executable, str(embed_script), "--db", str(database), "--checkpoint", str(checkpoint),\n                "--output", str(vectors_dir), "--gpu", str(gpu), "--gpus", "2", "--shards", "2",\n                "--batch", str(args.batch), "--onnx", str(model_dir / "seforim-embed-round2-int8.onnx"),\n            ]\n            if args.amp:\n                command.append("--amp")\n            processes.append(subprocess.Popen(command, env=env))\n        statuses = [process.wait() for process in processes]\n        if any(status != 0 for status in statuses):\n            raise RuntimeError(f"GPU encoding failed: {statuses}")\n        counts = [json.loads((vectors_dir / f"worker-{gpu}.json").read_text()) for gpu in range(2)]\n        print("GPU workers:", counts, flush=True)\n\n    library = args.repo / "SeforimLibrary"\n    if any(worker.get("vectorEncoding") != "int8-maxabs-v1" for worker in counts):\n        raise RuntimeError("GPU workers must produce int8 vectors")\n    gradle = library / "gradlew"\n    gradle.chmod(gradle.stat().st_mode | 0o111)\n    index.mkdir(exist_ok=True)\n    # Both GPUs write temporary vector streams; one writer builds the final index.\n    manifest = index / "shard-00" / "semantic.properties"\n    expected_vectors = sum(sum(worker["counts"].values()) for worker in counts)\n    properties = {}\n    if manifest.is_file():\n        content = manifest.read_text(encoding="utf-8", errors="ignore")\n        properties = dict(line.split("=", 1) for line in content.splitlines()\n                          if "=" in line and not line.startswith("#"))\n    complete = (\n        properties.get("format") == "zayit-round2-1"\n        and properties.get("shardIndex") == "0"\n        and properties.get("shardCount") == "1"\n        and properties.get("vectorEncoding") == "int8-maxabs-v1"\n        and properties.get("databaseSha256") == database_sha\n        and properties.get("modelSha256") == MODEL_SHA\n        and properties.get("tokenizerSha256") == TOKENIZER_SHA\n        and properties.get("dimension") == "256"\n        and expected_vectors > 0\n        and properties.get("indexed") == str(expected_vectors)\n        and properties.get("eligible") == str(expected_vectors)\n    )\n    if complete:\n        print("Skipping completed unified index", flush=True)\n    else:\n        for gpu, path in enumerate(vector_files):\n            expected_bytes = 12 + sum(counts[gpu]["counts"].values()) * (20 + 256)\n            if not path.is_file() or path.stat().st_size != expected_bytes:\n                raise ValueError(f"Missing or incomplete GPU vector file: {path}")\n        run_command([\n            str(gradle), ":search:buildSemanticIndexFromVectors",\n            f"-PseforimDb={database}", f"-PsemanticModelDir={model_dir}",\n            f"-PsemanticVectors={vectors_dir}", f"-PsemanticIndexDir={index}",\n            "-PshardIndex=0", "-PshardCount=1",\n            "--no-daemon", "--no-configuration-cache", "--max-workers=4",\n        ], cwd=library, env=env)\n    for path in vector_files:\n        path.unlink(missing_ok=True)\n\n    if args.all_distributions:\n        from kaggle_publish_distributions import build_distributions\n\n        build_distributions(args, database, model_dir, index, env, token, sha256, download,\n                            run_command, MultipartStream)\n        return\n\n    archive = args.output / "semantic-bundle.tar.zst"\n    run_command([\n        str(gradle), ":packaging:packageSemanticBundle",\n        f"-PseforimDb={database}", f"-PsemanticModelDir={model_dir}",\n        f"-PsemanticIndexDir={index}", f"-PsemanticBundleOutput={archive}", "-PzstdLevel=22",\n        "--no-daemon", "--no-configuration-cache", "--max-workers=4",\n    ], cwd=library, env=env)\n    outputs = sorted(args.output.glob("semantic-bundle.tar.zst*"))\n    manifest = args.output / "semantic-bundle.json"\n    outputs.append(manifest)\n    checksums = args.output / "checksums.sha256"\n    checksums.write_text("".join(f"{sha256(path)}  {path.name}\\n" for path in outputs), encoding="utf-8")\n    print("Ready for download:", *[str(path) for path in outputs + [checksums]], sep="\\n", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--repo", type=Path, required=True)\n    parser.add_argument("--db-release-tag", required=True)\n    parser.add_argument("--db-repository", default="arieldaniely/SeforimLibrary")\n    parser.add_argument("--all-distributions", action="store_true")\n    parser.add_argument("--publish-tag")\n    parser.add_argument("--pdf-release-tag")\n    parser.add_argument("--app-repository", default="arieldaniely/Zayit")\n    parser.add_argument("--work", type=Path, default=Path("/kaggle/temp/zayit-semantic"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working"))\n    parser.add_argument("--batch", type=int, default=128)\n    parser.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)\n    main(parser.parse_args())\n', 'scripts/kaggle_publish_distributions.py': '"""Package and publish six distributions sequentially, keeping archives out of saved Kaggle output."""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport uuid\nimport json\nimport re\nimport shutil\nimport tarfile\nfrom pathlib import Path\nfrom urllib.parse import quote\n\nimport requests\n\n\ndef extract_complete(parts, destination, stream_class):\n    """A completion marker prevents reuse of a partially extracted bundle."""\n    import zstandard\n\n    destination = destination.resolve()\n    marker = destination / \'.base-extracted\'\n    if marker.exists():\n        return\n    with contextlib.closing(stream_class(parts)) as source:\n        with zstandard.ZstdDecompressor().stream_reader(source) as reader:\n            with tarfile.open(fileobj=reader, mode=\'r|\') as archive:\n                for member in archive:\n                    target = (destination / member.name).resolve()\n                    if not target.is_relative_to(destination):\n                        raise ValueError(\'Archive entry escapes destination\')\n                    if member.isdir():\n                        target.mkdir(parents=True, exist_ok=True)\n                    elif member.isfile():\n                        target.parent.mkdir(parents=True, exist_ok=True)\n                        temporary = target.with_name(target.name + \'.extracting\')\n                        with archive.extractfile(member) as content, temporary.open(\'wb\') as output:\n                            shutil.copyfileobj(content, output, 1 << 20)\n                        temporary.replace(target)\n                    else:\n                        raise ValueError(\'Unsupported archive entry\')\n    marker.touch()\n\n\nclass Publisher:\n    def __init__(self, token):\n        self.session = requests.Session()\n        self.session.headers.update(Authorization=f\'Bearer {token}\', Accept=\'application/vnd.github+json\')\n\n    def api(self, method, path, **kwargs):\n        response = self.session.request(method, \'https://api.github.com/\' + path, timeout=(30, 120), **kwargs)\n        response.raise_for_status()\n        return response.json()\n\n    def draft(self, repository, tag, notes):\n        path = f\'repos/{repository}/releases\'\n        response = self.session.get(f\'https://api.github.com/{path}/tags/{quote(tag, safe="")}\', timeout=(30, 120))\n        if response.status_code == 404:\n            release = self.api(\'POST\', path, json={\'tag_name\': tag, \'name\': tag, \'body\': notes, \'draft\': True})\n        else:\n            response.raise_for_status()\n            release = response.json()\n        if not release[\'draft\']:\n            raise ValueError(f\'Release {repository}@{tag} is already published; choose a new tag\')\n        return release\n\n    def allocate(self, repository, app_repository, requested_tag, work):\n        """Resume our drafts, or allocate a fresh tag even if FULL already exists."""\n        state = work / \'publication.json\'\n        if state.exists():\n            saved = json.loads(state.read_text(encoding=\'utf-8\'))\n            if (saved[\'repository\'], saved[\'appRepository\']) != (repository, app_repository):\n                raise ValueError(\'Publication work directory belongs to another repository\')\n            releases = [self.api(\'GET\', f\'repos/{repo}/releases/{saved[key]}\')\n                        for repo, key in ((repository, \'releaseId\'), (app_repository, \'semanticReleaseId\'))]\n            if all(release[\'draft\'] for release in releases):\n                return saved[\'tag\']\n        # GitHub permits only one release per tag. A unique derived tag preserves existing releases.\n        while True:\n            tag = requested_tag + \'-\' + uuid.uuid4().hex[:12]\n            try:\n                release = self.draft(repository, tag, \'Building all six distributions\')\n                semantic = self.draft(app_repository, \'semantic-round2-\' + tag, \'Building semantic supplement\')\n                break\n            except requests.HTTPError as error:\n                if error.response.status_code != 422:\n                    raise\n        state.write_text(json.dumps({\'repository\': repository, \'appRepository\': app_repository,\n                                     \'tag\': tag, \'releaseId\': release[\'id\'],\n                                     \'semanticReleaseId\': semantic[\'id\']}), encoding=\'utf-8\')\n        print(\'Publication tag:\', tag, flush=True)\n        return tag\n\n    def upload(self, repository, release, file, digest):\n        for attempt in range(3):\n            try:\n                return self._upload(repository, release, file, digest)\n            except requests.RequestException:\n                if attempt == 2:\n                    raise\n                print(f\'Retrying upload: {file.name}\', flush=True)\n\n    def _upload(self, repository, release, file, digest):\n        # Recheck draft status before every replacement, including retries.\n        current = self.api(\'GET\', f\'repos/{repository}/releases/{release["id"]}\')\n        if not current[\'draft\']:\n            raise ValueError(\'Release was published while this build was running\')\n        for asset in current[\'assets\']:\n            if asset[\'name\'] != file.name:\n                continue\n            if asset.get(\'digest\') == f\'sha256:{digest}\' and asset[\'size\'] == file.stat().st_size:\n                return asset\n            response = self.session.delete(asset[\'url\'], timeout=(30, 120))\n            response.raise_for_status()\n        url = release[\'upload_url\'].split(\'{\', 1)[0]\n        with file.open(\'rb\') as source:\n            response = self.session.post(url, params={\'name\': file.name}, data=source,\n                                         headers={\'Content-Type\': \'application/octet-stream\'}, timeout=(30, 1800))\n        response.raise_for_status()\n        asset = response.json()\n        if asset.get(\'digest\') != f\'sha256:{digest}\' or asset[\'size\'] != file.stat().st_size:\n            raise ValueError(f\'Uploaded asset checksum mismatch: {file.name}\')\n        print(f\'Uploaded {repository}: {file.name}\', flush=True)\n        return asset\n\n    def publish(self, repository, release, latest):\n        return self.api(\'PATCH\', f\'repos/{repository}/releases/{release["id"]}\',\n                        json={\'draft\': False, \'prerelease\': False, \'make_latest\': str(latest).lower()})\n\n\ndef build_distributions(args, database, model_dir, index, env, token, sha256, download, run_command, stream_class):\n    publisher = Publisher(token)\n    tag = args.publish_tag or args.db_release_tag + \'-full\'\n    if tag == args.db_release_tag:\n        raise ValueError(\'Use a new final tag; the source release must remain unchanged\')\n    notes = f\'Six distributions for database {args.db_release_tag}. Database SHA-256: {sha256(database)}\'\n    release = publisher.draft(args.db_repository, tag, notes)\n    semantic_tag = \'semantic-round2-\' + tag\n    semantic_release = publisher.draft(args.app_repository, semantic_tag, notes)\n    work = args.work / \'distributions\'\n    work.mkdir(exist_ok=True)\n    pdf_response = requests.get(\n        \'https://api.github.com/repos/Otzaria/otzaria-library/releases/\' +\n        (f\'tags/{quote(args.pdf_release_tag, safe="")}\' if args.pdf_release_tag else \'latest\'), timeout=(30, 120),\n    )\n    pdf_response.raise_for_status()\n    pdf_release = pdf_response.json()\n    pdf_asset = next(asset for asset in pdf_release[\'assets\'] if asset[\'name\'] == \'talmud_bavli_latest.tar.zst\')\n    digest = pdf_asset.get(\'digest\') or \'\'\n    if not re.fullmatch(r\'sha256:[0-9a-f]{64}\', digest):\n        raise ValueError(\'PDF release asset has no SHA-256 digest\')\n    pdf_source = work / \'pdf-source.json\'\n    identity = {\'tag\': pdf_release[\'tag_name\'], \'digest\': digest}\n    if pdf_source.exists() and json.loads(pdf_source.read_text()) != identity:\n        raise ValueError(\'PDF source changed; pin --pdf-release-tag to the original PDF release\')\n    pdf_source.write_text(json.dumps(identity), encoding=\'utf-8\')\n    pdf_archive = work / pdf_asset[\'name\']\n    download(pdf_asset[\'browser_download_url\'], pdf_archive, digest.removeprefix(\'sha256:\'))\n    pdf_root = work / \'pdf\'\n    pdf_root.mkdir(exist_ok=True)\n    extract_complete([pdf_archive], pdf_root, stream_class)\n    pdf_dir = pdf_root / \'תלמוד בבלי\'\n    if not any(pdf_dir.rglob(\'*.pdf\')):\n        raise ValueError(\'PDF archive does not contain the expected תלמוד בבלי directory\')\n    pdf_archive.unlink()\n    for name in (\'seforim.db.lucene\', \'seforim.db.lookup.lucene\', \'catalog.pb\', \'lexical.db\', \'release_info.txt\'):\n        if not (database.parent / name).exists():\n            raise FileNotFoundError(f\'Missing base artifact: {name}\')\n    semantic_dir = work / \'semantic-single\'\n    semantic_dir.mkdir(exist_ok=True)\n    # Links avoid copying large indexes; the existing JVM packager traverses their contents.\n    for name, target in ((\'model\', model_dir), (\'index\', index)):\n        link = semantic_dir / name\n        if not link.exists():\n            link.symlink_to(target.resolve(), target_is_directory=True)\n    library = args.repo / \'SeforimLibrary\'\n    gradle = str(library / \'gradlew\')\n    entries = []\n    variants = [\n        (\'seforim_bundle-database-only\', False, False, False),\n        (\'seforim_bundle-no-vectors\', True, False, False),\n        (\'seforim_bundle-no-pdf\', False, True, False),\n        (\'seforim_bundle\', True, True, False),\n        (\'talmud_bavli_latest\', True, False, True),\n        (\'semantic-bundle\', False, True, False),\n    ]\n    for name, pdf, vectors, pdf_only in variants:\n        stage = work / name\n        stage.mkdir(exist_ok=True)\n        archive = stage / (name + \'.tar.zst\')\n        # A retry rebuilds this variant; remove only its prior generated archive files.\n        for old in stage.glob(archive.name + \'*\'):\n            if old.is_file():\n                old.unlink()\n        common = [\'--no-daemon\', \'--no-configuration-cache\', \'--max-workers=4\']\n        if name == \'semantic-bundle\':\n            command = [gradle, \':packaging:packageSemanticBundle\', f\'-PseforimDb={database}\',\n                       f\'-PsemanticModelDir={model_dir}\', f\'-PsemanticIndexDir={index}\',\n                       f\'-PsemanticBundleOutput={archive}\', \'-PzstdLevel=22\']\n        else:\n            command = [gradle, \':packaging:packageArtifacts\', f\'-PseforimDb={database}\',\n                       f\'-PpdfLibraryDir={pdf_dir}\', f\'-PsemanticBundleDir={semantic_dir}\',\n                       f\'-PincludePdf={str(pdf).lower()}\', f\'-PincludeVectors={str(vectors).lower()}\',\n                       f\'-PpdfOnly={str(pdf_only).lower()}\', f\'-PbundleOutput={archive}\', \'-PzstdLevel=22\',\n                       \'-x\', \':packaging:writeReleaseInfo\', \'-x\', \':packaging:downloadLexicalDb\']\n        run_command(command + common, cwd=library, env=env)\n        parts = sorted(stage.glob(archive.name + \'.part*\'))\n        files = parts or [archive]\n        if name == \'semantic-bundle\':\n            files.append(stage / \'semantic-bundle.json\')\n        for file in files:\n            if not file.is_file() or file.stat().st_size >= 2**31:\n                raise ValueError(f\'Asset missing or exceeds GitHub 2 GiB limit: {file.name}\')\n            digest = sha256(file)\n            asset = publisher.upload(args.db_repository, release, file, digest)\n            entries.append({\'name\': file.name, \'bytes\': file.stat().st_size, \'sha256\': digest,\n                            \'url\': asset[\'browser_download_url\']})\n            if name == \'semantic-bundle\':\n                publisher.upload(args.app_repository, semantic_release, file, digest)\n            file.unlink()\n        # The JVM base packager retains the unsplit archive alongside its parts.\n        archive.unlink(missing_ok=True)\n    manifest = args.output / \'distributions.json\'\n    manifest.write_text(json.dumps({\'databaseSha256\': sha256(database), \'databaseRelease\': args.db_release_tag,\n                                   \'compressionLevel\': 22, \'vectorEncoding\': \'int8-maxabs-v1\', \'shardCount\': 1,\n                                   \'releaseTag\': tag, \'pdfSource\': identity, \'defaultBundle\': \'seforim_bundle\', \'assets\': entries},\n                                  indent=2), encoding=\'utf-8\')\n    checksums = args.output / \'checksums.sha256\'\n    checksums.write_text(\'\'.join(f"{entry[\'sha256\']}  {entry[\'name\']}\\n" for entry in entries), encoding=\'utf-8\')\n    for file in (manifest, checksums):\n        publisher.upload(args.db_repository, release, file, sha256(file))\n    publisher.publish(args.app_repository, semantic_release, False)\n    publisher.publish(args.db_repository, release, True)\n    url = f\'https://github.com/{args.db_repository}/releases/tag/{tag}\'\n    (args.output / \'published-release.txt\').write_text(url + \'\\n\', encoding=\'utf-8\')\n    print(\'Published all six distributions:\', url, flush=True)\n', 'scripts/kaggle_embed_vectors.py': '"""Create Lucene input records from the Round 2 checkpoint on one Kaggle GPU."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport html\nimport json\nimport re\nimport sqlite3\nimport struct\nimport time\nimport unicodedata\nfrom html.parser import HTMLParser\nfrom pathlib import Path\n\nimport numpy as np\nimport regex\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoTokenizer, BertModel, PreTrainedTokenizerFast\n\n\nMARKS = re.compile(r"[\\u0591-\\u05BD\\u05BF-\\u05C7]")\nINVISIBLE = re.compile(r"[\\u200B-\\u200F\\u202A-\\u202E\\u2060-\\u2069\\uFEFF]")\nPUNCTUATION = re.compile(r"([!?.,;:])\\1{2,}")\nSPACE = re.compile(r"\\s+")\nTRANSLATION = str.maketrans({\n    "`": "\'", "´": "\'", "‘": "\'", "’": "\'", "‚": "\'", "‛": "\'", "׳": "\'",\n    "“": \'"\', "”": \'"\', "„": \'"\', "‟": \'"\', "״": \'"\', "־": "-", "–": "-",\n    "—": "-", "−": "-", "…": "...", "׃": ":",\n})\nBLOCK_TAGS = {\n    "address", "article", "aside", "blockquote", "br", "div", "footer", "h1", "h2",\n    "h3", "h4", "h5", "h6", "header", "li", "p", "section", "table", "td", "th", "tr",\n}\nVECTOR_DIMENSION = 256\nVECTOR_ENCODING = "int8-maxabs-v1"\nFILE_HEADER = b"ZYVECI8\\n" + struct.pack("<I", VECTOR_DIMENSION)\nRECORD_BYTES = 8 + 8 + 4 + VECTOR_DIMENSION\n\n\ndef quantize_vectors(vectors: np.ndarray) -> np.ndarray:\n    """Match Int8Vectors.quantize; discard per-vector scale for cosine search."""\n    vectors = np.asarray(vectors, dtype=np.float32)\n    if vectors.ndim != 2 or vectors.shape[1] != VECTOR_DIMENSION or not np.isfinite(vectors).all():\n        raise ValueError("Invalid semantic vectors")\n    maximum = np.max(np.abs(vectors), axis=1, keepdims=True)\n    if np.any(maximum == 0):\n        raise ValueError("Zero semantic vector")\n    return np.clip(np.floor(vectors / maximum * np.float32(127) + np.float32(0.5)), -127, 127).astype(np.int8)\n\n\nclass VisibleText(HTMLParser):\n    def __init__(self) -> None:\n        super().__init__(convert_charrefs=True)\n        self.parts: list[str] = []\n        self.hidden = 0\n\n    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:\n        if tag in {"script", "style", "template"}:\n            self.hidden += 1\n        elif tag in BLOCK_TAGS:\n            self.parts.append(" ")\n\n    def handle_endtag(self, tag: str) -> None:\n        if tag in {"script", "style", "template"} and self.hidden:\n            self.hidden -= 1\n        elif tag in BLOCK_TAGS:\n            self.parts.append(" ")\n\n    def handle_data(self, data: str) -> None:\n        if not self.hidden:\n            self.parts.append(data)\n\n\ndef clean(text: str) -> str:\n    if not text:\n        return ""\n    if "<" in text or "&" in text:\n        parser = VisibleText()\n        parser.feed(text)\n        parser.close()\n        text = "".join(parser.parts)\n    text = unicodedata.normalize("NFKC", html.unescape(text))\n    text = INVISIBLE.sub("", MARKS.sub("", text)).translate(TRANSLATION)\n    text = regex.sub(r"\\p{Script=Latin}+", " ", text)\n    return SPACE.sub(" ", PUNCTUATION.sub(r"\\1\\1", text)).strip()\n\n\nclass SentenceEncoder(nn.Module):\n    def __init__(self, backbone: BertModel, dimension: int) -> None:\n        super().__init__()\n        self.backbone = backbone\n        self.projection = nn.Linear(backbone.config.hidden_size, dimension, bias=False)\n        self.projection_norm = nn.LayerNorm(dimension)\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n        mask = attention_mask.unsqueeze(-1).to(output.dtype)\n        pooled = (output * mask).sum(1) / mask.sum(1).clamp_min(1)\n        return F.normalize(self.projection_norm(self.projection(pooled)).float(), p=2, dim=1)\n\n\ndef load_model(checkpoint: Path, device: torch.device) -> tuple[PreTrainedTokenizerFast, SentenceEncoder]:\n    metadata = json.loads((checkpoint / "sentence_encoder_config.json").read_text(encoding="utf-8"))\n    assert int(metadata["embedding_dim"]) == 256\n    assert metadata["passage_prefix"] == "[PASSAGE]"\n    tokenizer_file = checkpoint / "tokenizer" / "tokenizer.json"\n    tokenizer = PreTrainedTokenizerFast(tokenizer_file=str(tokenizer_file))\n    vocab = tokenizer.get_vocab()\n    for name, candidate in (("pad_token", "[PAD]"), ("unk_token", "[UNK]"),\n                            ("cls_token", "[CLS]"), ("sep_token", "[SEP]"),\n                            ("mask_token", "[MASK]")):\n        if getattr(tokenizer, name) is None and candidate in vocab:\n            setattr(tokenizer, name, candidate)\n    backbone = BertModel.from_pretrained(checkpoint / "backbone")\n    model = SentenceEncoder(backbone, 256)\n    weights = load_file(str(checkpoint / "projection.safetensors"), device="cpu")\n    model.projection.load_state_dict({"weight": weights["projection.weight"]})\n    model.projection_norm.load_state_dict({\n        "weight": weights["projection_norm.weight"],\n        "bias": weights["projection_norm.bias"],\n    })\n    return tokenizer, model.to(device).eval()\n\n\ndef run(args: argparse.Namespace) -> None:\n    assert args.shards > 0 and args.shards % args.gpus == 0\n    assert 0 <= args.gpu < args.gpus\n    torch.set_num_threads(2)\n    device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")\n    if device.type == "cuda":\n        assert torch.cuda.device_count() >= args.gpus, "Select Kaggle T4 x2 accelerator"\n    print(f"worker {args.gpu}: device={device}", flush=True)\n    tokenizer, model = load_model(args.checkpoint, device)\n    args.output.mkdir(parents=True, exist_ok=True)\n    counts = {shard: 0 for shard in range(args.gpu, args.shards, args.gpus)}\n    outputs = {shard: (args.output / f"shard-{shard:02d}.bin").open("wb", buffering=1 << 20)\n               for shard in counts}\n    for output in outputs.values():\n        output.write(FILE_HEADER)\n    connection = sqlite3.connect(f"file:{args.db}?mode=ro", uri=True)\n    started = time.monotonic()\n    validated = False\n\n    @torch.inference_mode()\n    def encode_batch(batch: list[tuple[int, int, int, str]]) -> None:\n        nonlocal validated\n        tokens = tokenizer(\n            [f"[PASSAGE] {row[3]}" for row in batch], padding=True, truncation=True,\n            max_length=256, return_tensors="pt",\n        )\n        token_inputs = {key: tokens[key].to(device) for key in ("input_ids", "attention_mask")}\n        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda" and args.amp):\n            vectors = model(**token_inputs).cpu().numpy().astype("<f4", copy=False)\n        assert vectors.shape == (len(batch), 256) and np.isfinite(vectors).all()\n        if args.gpu == 0 and args.onnx is not None and not validated:\n            import onnxruntime as ort\n\n            session = ort.InferenceSession(str(args.onnx), providers=["CPUExecutionProvider"])\n            cosine_values = []\n            for position in range(min(8, len(batch))):\n                reference = session.run(None, {\n                    "input_ids": token_inputs["input_ids"][position:position + 1].cpu().numpy(),\n                    "attention_mask": token_inputs["attention_mask"][position:position + 1].cpu().numpy(),\n                })[0][0]\n                cosine_values.append(float(np.dot(reference, vectors[position]) /\n                                           (np.linalg.norm(reference) * np.linalg.norm(vectors[position]))))\n            print(f"GPU/ONNX minimum cosine: {min(cosine_values):.6f}", flush=True)\n            assert min(cosine_values) >= 0.985, "GPU vectors do not match the bundled ONNX model"\n            validated = True\n        for (line_id, book_id, is_base, _), vector in zip(batch, quantize_vectors(vectors)):\n            shard = line_id % args.shards\n            outputs[shard].write(struct.pack("<qqi", line_id, book_id, is_base))\n            outputs[shard].write(vector.tobytes())\n            counts[shard] += 1\n\n    try:\n        rows = connection.execute(\n            "SELECT l.id, l.bookId, l.content, b.isBaseBook FROM line l "\n            "JOIN book b ON b.id = l.bookId WHERE l.id % ? = ? ORDER BY l.id",\n            (args.gpus, args.gpu),\n        )\n        batch: list[tuple[int, int, int, str]] = []\n        scanned = 0\n        for line_id, book_id, content, is_base in rows:\n            scanned += 1\n            normalized = clean(content or "")\n            if normalized:\n                batch.append((line_id, book_id, is_base, normalized))\n            if len(batch) >= args.batch:\n                encode_batch(batch)\n                batch.clear()\n            if scanned % 100_000 == 0:\n                print(f"worker {args.gpu}: scanned {scanned}, indexed {sum(counts.values())}, "\n                      f"elapsed {time.monotonic() - started:.0f}s", flush=True)\n        if batch:\n            encode_batch(batch)\n    finally:\n        connection.close()\n        for output in outputs.values():\n            output.close()\n    for shard, count in counts.items():\n        assert (args.output / f"shard-{shard:02d}.bin").stat().st_size == len(FILE_HEADER) + count * RECORD_BYTES\n    (args.output / f"worker-{args.gpu}.json").write_text(\n        json.dumps({"scanned": scanned, "counts": counts, "vectorEncoding": VECTOR_ENCODING}, indent=2) + "\\n", encoding="utf-8",\n    )\n    print(f"worker {args.gpu}: completed {sum(counts.values())} vectors", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--db", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--gpu", type=int, required=True)\n    parser.add_argument("--gpus", type=int, default=2)\n    # Temporary GPU streams are combined into one Lucene index by the bundle builder.\n    parser.add_argument("--shards", type=int, default=2)\n    parser.add_argument("--batch", type=int, default=128)\n    parser.add_argument("--amp", action="store_true")\n    parser.add_argument("--onnx", type=Path)\n    run(parser.parse_args())\n', 'SeforimLibrary/generator/packaging/build.gradle.kts': 'plugins {\n    alias(libs.plugins.multiplatform)\n}\n\nkotlin {\n    jvmToolchain(libs.versions.jvmToolchain.get().toInt())\n\n    jvm()\n\n    sourceSets {\n        commonMain.dependencies {\n            implementation(libs.kermit)\n        }\n\n        commonTest.dependencies {\n            implementation(kotlin("test"))\n        }\n\n        jvmMain.dependencies {\n            implementation(project(":generator-common"))\n            implementation(project(":search"))\n            implementation(libs.zstd)\n            implementation(libs.commons.compress)\n        }\n    }\n}\n\ntasks.register<JavaExec>("packageSemanticBundle") {\n    group = "application"\n    description = "Package a complete local Round 2 index and model as tar.zst, splitting only when needed"\n    workingDir = layout.projectDirectory.asFile\n    dependsOn("jvmJar")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.packaging.PackageSemanticBundleKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n    val db = providers.gradleProperty("seforimDb").orElse(providers.environmentVariable("SEFORIM_DB"))\n    val model = providers.gradleProperty("semanticModelDir")\n    val index = providers.gradleProperty("semanticIndexDir")\n    val output = providers.gradleProperty("semanticBundleOutput")\n    argumentProviders.add(org.gradle.process.CommandLineArgumentProvider {\n        listOf(db.get(), model.get(), index.get(), output.get(),\n            providers.gradleProperty("splitPartBytes").orElse("2040109465").get(),\n            providers.gradleProperty("zstdLevel").orElse("22").get())\n    })\n    maxHeapSize = "2g"\n}\n\n// Write release information to release_info.txt file\n// Usage:\n//   ./gradlew :packaging:writeReleaseInfo\n//   ./gradlew :packaging:writeReleaseInfo -PreleaseName=20251108195010\n//   ./gradlew :packaging:writeReleaseInfo -PseforimDb=/path/to/seforim.db\ntasks.register<JavaExec>("writeReleaseInfo") {\n    group = "application"\n    description = "Write release information (timestamp) to release_info.txt file."\n\n    dependsOn("jvmJar")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.packaging.WriteReleaseInfoKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n\n    // Ensure release_info.txt is written next to the same DB that will be packaged.\n    if (project.hasProperty("seforimDb")) {\n        systemProperty("seforimDb", project.property("seforimDb") as String)\n    } else if (System.getenv("SEFORIM_DB") != null) {\n        systemProperty("seforimDb", System.getenv("SEFORIM_DB"))\n    } else {\n        val defaultDbPath = rootProject.layout.buildDirectory.file("seforim.db").get().asFile.absolutePath\n        systemProperty("seforimDb", defaultDbPath)\n    }\n\n    // Pass release name if provided\n    if (project.hasProperty("releaseName")) {\n        systemProperty("releaseName", project.property("releaseName") as String)\n    }\n\n    jvmArgs = listOf("-Xmx256m")\n}\n\n// Download lexical.db (from latest SeforimMagicIndexer release) next to seforim.db\n// Usage:\n//   ./gradlew :packaging:downloadLexicalDb\n//   ./gradlew :packaging:downloadLexicalDb -PseforimDb=/path/to/seforim.db\ntasks.register<JavaExec>("downloadLexicalDb") {\n    group = "application"\n    description = "Download lexical.db from latest SeforimMagicIndexer release next to seforim.db."\n\n    dependsOn("jvmJar")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.packaging.DownloadLexicalDbKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n\n    // Place lexical.db next to the same DB that will be packaged.\n    if (project.hasProperty("seforimDb")) {\n        systemProperty("seforimDb", project.property("seforimDb") as String)\n    } else if (System.getenv("SEFORIM_DB") != null) {\n        systemProperty("seforimDb", System.getenv("SEFORIM_DB"))\n    } else {\n        val defaultDbPath = rootProject.layout.buildDirectory.file("seforim.db").get().asFile.absolutePath\n        systemProperty("seforimDb", defaultDbPath)\n    }\n\n    jvmArgs = listOf("-Xmx512m")\n}\n\n// Package DB + Lucene indexes into single tar.zst and split\ntasks.register<JavaExec>("packageArtifacts") {\n    group = "application"\n    description = "Create seforim_bundle.tar.zst (DB + indexes + release info) with zstd and split into ~1.9GiB parts."\n\n    dependsOn("jvmJar", "writeReleaseInfo", "downloadLexicalDb")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.packaging.PackageArtifactsKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n\n    // Pass optional properties if provided\n    if (project.hasProperty("seforimDb")) {\n        systemProperty("seforimDb", project.property("seforimDb") as String)\n    } else if (System.getenv("SEFORIM_DB") != null) {\n        systemProperty("seforimDb", System.getenv("SEFORIM_DB"))\n    } else {\n        // Default to DB under root build/\n        val defaultDbPath = rootProject.layout.buildDirectory.file("seforim.db").get().asFile.absolutePath\n        systemProperty("seforimDb", defaultDbPath)\n    }\n\n    listOf("pdfOnly", "includePdf", "includeVectors", "pdfLibraryDir", "semanticBundleDir").forEach { name ->\n        providers.gradleProperty(name).orNull?.let { systemProperty(name, it) }\n    }\n\n    // Output bundle\n    if (project.hasProperty("bundleOutput")) {\n        systemProperty("bundleOutput", project.property("bundleOutput") as String)\n    }\n    // Backward-compatible: if -Poutput was provided, pass it through as legacy\n    if (project.hasProperty("output")) {\n        systemProperty("output", project.property("output") as String)\n    }\n    if (project.hasProperty("zstdLevel")) {\n        systemProperty("zstdLevel", project.property("zstdLevel") as String)\n    }\n    if (project.hasProperty("zstdWorkers")) {\n        systemProperty("zstdWorkers", project.property("zstdWorkers") as String)\n    }\n    if (project.hasProperty("splitPartBytes")) {\n        systemProperty("splitPartBytes", project.property("splitPartBytes") as String)\n    }\n\n    jvmArgs = listOf("-Xmx512m")\n}\n', 'SeforimLibrary/generator/packaging/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/packaging/PackageArtifacts.kt': 'package io.github.kdroidfilter.seforimlibrary.packaging\n\nimport co.touchlab.kermit.Logger\nimport co.touchlab.kermit.Severity\nimport com.github.luben.zstd.ZstdOutputStream\nimport io.github.kdroidfilter.seforimlibrary.search.VectorSearcher\nimport org.apache.commons.compress.archivers.tar.TarArchiveEntry\nimport org.apache.commons.compress.archivers.tar.TarArchiveOutputStream\nimport java.io.BufferedOutputStream\nimport java.io.BufferedInputStream\nimport java.io.IOException\nimport java.nio.file.Files\nimport java.nio.file.Path\nimport java.nio.file.Paths\nimport java.time.Instant\nimport java.util.Date\nimport kotlin.io.path.exists\nimport kotlin.system.exitProcess\n\n/**\n * Package artifacts with Zstandard (zstd):\n *  - SQLite DB (seforim.db)\n *  - Precomputed catalog (catalog.pb)\n *  - Lucene indexes (text and lookup)\n *  - Lexical DB (lexical.db)\n *  All packaged into a single .tar.zst bundle\n *\n * Usage:\n *   ./gradlew -p SeforimLibrary :packaging:packageArtifacts \\\n *     -PseforimDb=/path/to/seforim.db \\\n *     [-PbundleOutput=/path/to/seforim_bundle.tar.zst] \\\n *     [-PzstdLevel=22] \\\n *     [-PzstdWorkers=0] \\\n *     [-PsplitPartBytes=2040109465]\n *\n * Env alternatives:\n *   SEFORIM_DB, OUTPUT_BUNDLE_TAR_ZST, ZSTD_LEVEL, ZSTD_WORKERS, SPLIT_PART_BYTES\n *   (legacy: OUTPUT_TAR_ZST or -Poutput maps to bundleOutput for compatibility)\n *\n * Defaults:\n *   DB path from -PseforimDb, env SEFORIM_DB, or build/seforim.db\n *   Bundle output to build/package/seforim_bundle.tar.zst\n *   Catalog expected at same location as DB (catalog.pb)\n *   Indexes expected next to DB (.lucene, .lookup.lucene)\n *   Release info expected next to DB (release_info.txt)\n *   Lexical DB expected next to DB (lexical.db)\n */\nfun main(args: Array<String>) {\n    Logger.setMinSeverity(Severity.Info)\n    val logger = Logger.withTag("PackageArtifacts")\n\n    // Resolve DB path\n    val dbPathStr = args.getOrNull(0)\n        ?: System.getProperty("seforimDb")\n        ?: System.getenv("SEFORIM_DB")\n        ?: Paths.get("build", "seforim.db").toString()\n    val dbPath = Paths.get(dbPathStr)\n    if (!dbPath.exists()) {\n        logger.e { "DB not found at $dbPath" }\n        exitProcess(1)\n    }\n\n    // Resolve index directories next to the DB\n    val textIndexDir: Path = if (dbPathStr.endsWith(".db")) Paths.get("$dbPathStr.lucene") else Paths.get("$dbPathStr.luceneindex")\n    val lookupIndexDir: Path = if (dbPathStr.endsWith(".db")) Paths.get("$dbPathStr.lookup.lucene") else Paths.get("$dbPathStr.lookupindex")\n\n    // Resolve precomputed catalog next to the DB\n    val catalogPath: Path = dbPath.resolveSibling("catalog.pb")\n\n    // Resolve release info file next to the DB\n    val releaseInfoPath: Path = dbPath.resolveSibling("release_info.txt")\n\n    // Resolve lexical DB next to the DB\n    val lexicalDbPath: Path = dbPath.resolveSibling("lexical.db")\n\n    if (!textIndexDir.toFile().isDirectory) {\n        logger.w { "Lucene text index directory missing: $textIndexDir (will skip)" }\n    }\n    if (!lookupIndexDir.toFile().isDirectory) {\n        logger.w { "Lucene lookup index directory missing: $lookupIndexDir (will skip)" }\n    }\n    if (!catalogPath.exists()) {\n        logger.w { "Precomputed catalog missing: $catalogPath (will skip)" }\n    }\n    if (!releaseInfoPath.exists()) {\n        logger.w { "Release info file missing: $releaseInfoPath (will skip)" }\n    }\n    if (!lexicalDbPath.exists()) {\n        logger.w { "Lexical DB missing: $lexicalDbPath (will skip)" }\n    }\n\n    val pdfOnly = System.getProperty("pdfOnly", "false").toBooleanStrict()\n    val includePdf = System.getProperty("includePdf", "true").toBooleanStrict()\n    val includeVectors = System.getProperty("includeVectors", "true").toBooleanStrict()\n    val pdfDir = Paths.get(System.getProperty("pdfLibraryDir") ?: dbPath.resolveSibling("תלמוד בבלי").toString())\n    val semanticDir = Paths.get(System.getProperty("semanticBundleDir") ?: "$dbPathStr.semantic")\n    require(!includePdf || pdfDir.toFile().walkTopDown().any { it.isFile && it.extension.equals("pdf", true) }) {\n        "PDF library missing; provide -PpdfLibraryDir or explicitly use -PincludePdf=false"\n    }\n    require(pdfOnly || !includeVectors || (\n        Files.isRegularFile(semanticDir.resolve("model/seforim-embed-round2-int8.onnx")) &&\n            Files.isRegularFile(semanticDir.resolve("model/tokenizer.json")) &&\n            Files.isDirectory(semanticDir.resolve("index"))\n    )) { "Semantic bundle missing; provide -PsemanticBundleDir or explicitly use -PincludeVectors=false" }\n\n    if (!pdfOnly && includeVectors) {\n        VectorSearcher(semanticDir.resolve("index"), 256, dbPath, semanticDir.resolve("model")).use { }\n    }\n\n    // Output: single bundle tar.zst\n    val legacyOutput = System.getProperty("output") ?: System.getenv("OUTPUT_TAR_ZST")\n    val bundleOutputStr = System.getProperty("bundleOutput")\n        ?: System.getenv("OUTPUT_BUNDLE_TAR_ZST")\n        ?: legacyOutput\n        ?: Paths.get("build", "package", "seforim_bundle.tar.zst").toString()\n    val bundleOutputPath = Paths.get(bundleOutputStr)\n    Files.createDirectories(bundleOutputPath.parent)\n\n    // Compression level (default ultra 22)\n    val zstdLevel = (\n        System.getProperty("zstdLevel")\n            ?: System.getenv("ZSTD_LEVEL")\n            ?: "22"\n        ).toIntOrNull()?.coerceIn(1, 22) ?: 22\n\n    // Workers (threads). Default: all available processors (like zstd -T0)\n    val workers = (\n        System.getProperty("zstdWorkers")\n            ?: System.getenv("ZSTD_WORKERS")\n            ?: "0"\n    ).toIntOrNull()?.let { if (it <= 0) Runtime.getRuntime().availableProcessors() else it }\n        ?: Runtime.getRuntime().availableProcessors()\n\n    // Split part size (~1.9 GiB by default)\n    val defaultSplitSize = (1.9 * 1024.0 * 1024.0 * 1024.0).toLong()\n    val splitPartBytes = (\n        System.getProperty("splitPartBytes")\n            ?: System.getenv("SPLIT_PART_BYTES")\n            ?: defaultSplitSize.toString()\n        ).toLongOrNull()?.takeIf { it > 0 } ?: defaultSplitSize\n\n    logger.i {\n        "Packaging into single bundle:\\n" +\n            " - DB: $dbPath\\n" +\n            " - Catalog: $catalogPath\\n" +\n            " - Release info: $releaseInfoPath\\n" +\n            " - Lexical DB: $lexicalDbPath\\n" +\n            " - Text index: $textIndexDir\\n" +\n            " - Lookup index: $lookupIndexDir\\n" +\n            " -> Bundle .tar.zst: $bundleOutputPath\\n" +\n            " (zstd level $zstdLevel, workers $workers, split ${humanSize(splitPartBytes)})"\n    }\n\n    try {\n        // Tar + zstd the three artifacts into a single bundle\n        Files.newOutputStream(bundleOutputPath).use { fos ->\n            BufferedOutputStream(fos, 1 shl 20).use { bos ->\n                ZstdOutputStream(bos, zstdLevel).use { zstd ->\n                    runCatching { zstd.setWorkers(workers) }\n                        .onFailure { logger.w(it) { "zstd setWorkers($workers) failed; continuing single-threaded" } }\n                    TarArchiveOutputStream(zstd).use { tar ->\n                        tar.setLongFileMode(TarArchiveOutputStream.LONGFILE_POSIX)\n                        tar.setBigNumberMode(TarArchiveOutputStream.BIGNUMBER_POSIX)\n\n                        if (!pdfOnly) {\n                            val haveText = textIndexDir.toFile().isDirectory\n                            val haveLookup = lookupIndexDir.toFile().isDirectory\n                            val haveCatalog = catalogPath.exists()\n                            val haveReleaseInfo = releaseInfoPath.exists()\n                            val haveLexicalDb = lexicalDbPath.exists()\n\n                            if (haveLookup) {\n                                addDirectoryToTar(tar, lookupIndexDir, lookupIndexDir.fileName.toString(), logger)\n                            } else {\n                                logger.w { "Lucene lookup index directory missing: $lookupIndexDir (skipped)" }\n                            }\n                            if (haveText) {\n                                addDirectoryToTar(tar, textIndexDir, textIndexDir.fileName.toString(), logger)\n                            } else {\n                                logger.w { "Lucene text index directory missing: $textIndexDir (skipped)" }\n                            }\n\n                            // Add the database file itself\n                            addFileToTar(tar, dbPath, dbPath.fileName.toString(), logger)\n\n                            // Add lexical DB if available\n                            if (haveLexicalDb) {\n                                addFileToTar(tar, lexicalDbPath, lexicalDbPath.fileName.toString(), logger)\n                                logger.i { "Added lexical DB to bundle" }\n                            } else {\n                                logger.w { "Lexical DB missing: $lexicalDbPath (skipped)" }\n                            }\n\n                            // Add the precomputed catalog if available\n                            if (haveCatalog) {\n                                addFileToTar(tar, catalogPath, catalogPath.fileName.toString(), logger)\n                                logger.i { "Added precomputed catalog to bundle" }\n                            } else {\n                                logger.w { "Precomputed catalog missing: $catalogPath (skipped)" }\n                            }\n\n                            // Add the release info file if available\n                            if (haveReleaseInfo) {\n                                addFileToTar(tar, releaseInfoPath, releaseInfoPath.fileName.toString(), logger)\n                                logger.i { "Added release info to bundle" }\n                            } else {\n                                logger.w { "Release info file missing: $releaseInfoPath (skipped)" }\n                            }\n\n                        }\n                        if (includePdf) addDirectoryToTar(tar, pdfDir, "תלמוד בבלי", logger)\n                        if (!pdfOnly && includeVectors) addDirectoryToTar(tar, semanticDir, "${dbPath.fileName}.semantic", logger)\n                        tar.finish()\n                    }\n                }\n            }\n        }\n\n        val bundleSizeMb = Files.size(bundleOutputPath).toDouble() / (1024 * 1024)\n        logger.i { "Bundle written: $bundleOutputPath (${"%.2f".format(bundleSizeMb)} MB)" }\n        println(bundleOutputPath.toAbsolutePath().toString())\n\n        // Split the bundle into parts of ~1.9 GiB (or configured size)\n        val partCount = splitFile(bundleOutputPath, splitPartBytes, logger)\n        if (partCount > 1) {\n            logger.i { "Bundle split into $partCount parts of up to ${humanSize(splitPartBytes)} each" }\n        } else {\n            logger.i { "Bundle size below split threshold; no split files created" }\n        }\n    } catch (e: Exception) {\n        logger.e(e) { "Failed to package artifacts" }\n        exitProcess(1)\n    }\n}\n\nprivate fun addDirectoryToTar(tar: TarArchiveOutputStream, dir: Path, entryName: String, logger: Logger) {\n    val normalized = entryName.trimEnd(\'/\')\n    val dirEntry = TarArchiveEntry("$normalized/")\n    dirEntry.modTime = Date.from(Instant.now())\n    try {\n        tar.putArchiveEntry(dirEntry)\n        tar.closeArchiveEntry()\n    } catch (e: IOException) {\n        logger.w(e) { "Skipping directory entry $normalized/ due to error" }\n        return\n    }\n\n    Files.list(dir).use { stream ->\n        stream.forEach { child ->\n            val childName = "$normalized/${child.fileName}"\n            if (Files.isDirectory(child)) {\n                addDirectoryToTar(tar, child, childName, logger)\n            } else {\n                addFileToTar(tar, child, childName, logger)\n            }\n        }\n    }\n}\n\nprivate fun addFileToTar(tar: TarArchiveOutputStream, file: Path, entryName: String, logger: Logger) {\n    val f = file.toFile()\n    if (!f.exists() || !f.isFile) {\n        logger.w { "Skipping missing file: $file" }\n        return\n    }\n    val entry = TarArchiveEntry(f, entryName)\n    // Ensure size is set for some JVMs\n    entry.size = f.length()\n    entry.modTime = Date(f.lastModified())\n    tar.putArchiveEntry(entry)\n    val declared = entry.size\n    Files.newInputStream(file).use { input ->\n        val buffer = ByteArray(1 shl 20)\n        var remaining = declared\n        while (remaining > 0) {\n            val toRead = if (remaining > buffer.size) buffer.size else remaining.toInt()\n            val read = input.read(buffer, 0, toRead)\n            if (read <= 0) break\n            tar.write(buffer, 0, read)\n            remaining -= read\n        }\n    }\n    tar.closeArchiveEntry()\n}\n\nprivate fun splitFile(source: Path, partSizeBytes: Long, logger: Logger): Int {\n    val size = try { Files.size(source) } catch (e: IOException) { 0L }\n    if (size <= partSizeBytes || size <= 0L) return 1\n\n    val parent = source.parent ?: Paths.get(".")\n    val baseName = source.fileName.toString()\n\n    val buffer = ByteArray(8 * 1024 * 1024) // 8 MiB\n    var index = 1\n    var bytesInPart = 0L\n    var currentOut: BufferedOutputStream? = null\n\n    fun openNext(): BufferedOutputStream {\n        val suffix = String.format(".part%02d", index)\n        val outPath = parent.resolve(baseName + suffix)\n        val bos = BufferedOutputStream(Files.newOutputStream(outPath), buffer.size)\n        logger.i { "Writing part $index -> ${outPath.toAbsolutePath()}" }\n        return bos\n    }\n\n    BufferedInputStream(Files.newInputStream(source), buffer.size).use { input ->\n        currentOut = openNext()\n        while (true) {\n            val remainingInPart = partSizeBytes - bytesInPart\n            val toRead = if (remainingInPart < buffer.size) remainingInPart.toInt() else buffer.size\n            val read = input.read(buffer, 0, toRead)\n            if (read <= 0) break\n            currentOut!!.write(buffer, 0, read)\n            bytesInPart += read\n            if (bytesInPart >= partSizeBytes) {\n                currentOut!!.flush()\n                currentOut!!.close()\n                index += 1\n                bytesInPart = 0\n                currentOut = openNext()\n            }\n        }\n    }\n    currentOut?.flush()\n    currentOut?.close()\n    return index\n}\n\nprivate fun humanSize(bytes: Long): String {\n    val kb = 1024.0\n    val mb = kb * 1024\n    val gb = mb * 1024\n    return when {\n        bytes >= gb -> String.format("%.2f GiB", bytes / gb)\n        bytes >= mb -> String.format("%.2f MiB", bytes / mb)\n        bytes >= kb -> String.format("%.2f KiB", bytes / kb)\n        else -> "$bytes B"\n    }\n}\n', 'SeforimLibrary/search/build.gradle.kts': 'import org.gradle.process.CommandLineArgumentProvider\n\nplugins {\n    alias(libs.plugins.multiplatform)\n    alias(libs.plugins.kotlinx.serialization)\n}\n\ntasks.register<JavaExec>("buildSemanticIndex") {\n    group = "application"\n    description = "Build one optional Round 2 semantic Lucene shard"\n    dependsOn("jvmJar")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.search.BuildSemanticIndexKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n    val db = providers.gradleProperty("seforimDb").orElse(providers.environmentVariable("SEFORIM_DB"))\n    val model = providers.gradleProperty("semanticModelDir")\n    val output = providers.gradleProperty("semanticIndexDir")\n    argumentProviders.add(CommandLineArgumentProvider {\n        listOf(db.get(), model.get(), output.get(),\n            providers.gradleProperty("shardIndex").orElse("0").get(),\n            providers.gradleProperty("shardCount").orElse("1").get(),\n            providers.gradleProperty("semanticWorkers").orElse("1").get())\n    })\n    providers.gradleProperty("semanticThreads").orNull?.let { systemProperty("seforimEmbedThreads", it) }\n    maxHeapSize = "4g"\n}\n\ntasks.register<JavaExec>("buildSemanticIndexFromVectors") {\n    group = "application"\n    description = "Build a unified semantic Lucene index from GPU-generated vector files"\n    dependsOn("jvmJar")\n    mainClass.set("io.github.kdroidfilter.seforimlibrary.search.BuildSemanticIndexFromVectorsKt")\n    classpath = files(tasks.named("jvmJar")) + configurations.getByName("jvmRuntimeClasspath")\n    workingDir = layout.projectDirectory.asFile\n    val db = providers.gradleProperty("seforimDb")\n    val model = providers.gradleProperty("semanticModelDir")\n    val vectors = providers.gradleProperty("semanticVectors")\n    val output = providers.gradleProperty("semanticIndexDir")\n    argumentProviders.add(CommandLineArgumentProvider {\n        listOf(db.get(), model.get(), vectors.get(), output.get(),\n            providers.gradleProperty("shardIndex").orElse("0").get(),\n            providers.gradleProperty("shardCount").orElse("1").get())\n    })\n    maxHeapSize = "4g"\n}\n\ngroup = "io.github.kdroidfilter.seforimlibrary"\n\nkotlin {\n    jvmToolchain(libs.versions.jvmToolchain.get().toInt())\n\n    jvm()\n\n    sourceSets {\n        jvmMain.dependencies {\n            api(project(":core"))\n            implementation(libs.kotlinx.coroutines.core)\n            implementation(libs.kotlinx.serialization.json)\n            implementation(libs.lucene.core)\n            implementation(libs.lucene.analysis.common)\n            implementation(libs.sqlDelight.driver.sqlite)\n            implementation(libs.kermit)\n            implementation(libs.jsoup)\n            // Dense semantic search: ONNX Runtime (query embedding) + HuggingFace tokenizer.\n            // Stock Maven `onnxruntime` is CPU-only on desktop JVM (no DirectML/CoreML/XNNPACK\n            // native — those need a custom build; CUDA needs the separate onnxruntime_gpu).\n            implementation(libs.onnxruntime)\n            implementation(libs.djl.huggingface.tokenizers)\n        }\n\n        jvmTest.dependencies {\n            implementation(kotlin("test"))\n            implementation(libs.kotlinx.coroutines.test)\n        }\n    }\n}\n', 'SeforimLibrary/search/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/search/BuildSemanticIndexFromVectors.kt': 'package io.github.kdroidfilter.seforimlibrary.search\n\nimport org.apache.lucene.analysis.standard.StandardAnalyzer\nimport org.apache.lucene.document.Document\nimport org.apache.lucene.document.IntPoint\nimport org.apache.lucene.document.KnnByteVectorField\nimport org.apache.lucene.document.StoredField\nimport org.apache.lucene.index.IndexWriter\nimport org.apache.lucene.index.IndexWriterConfig\nimport org.apache.lucene.index.VectorSimilarityFunction\nimport org.apache.lucene.store.FSDirectory\nimport java.io.BufferedInputStream\nimport java.io.DataInputStream\nimport java.nio.ByteBuffer\nimport java.nio.ByteOrder\nimport java.nio.file.Files\nimport java.nio.file.Path\nimport java.util.Properties\n\nprivate const val VECTOR_DIMENSION = 256\nprivate const val FLOAT_RECORD_BYTES = 8 + 8 + 4 + VECTOR_DIMENSION * 4\nprivate const val INT8_RECORD_BYTES = 8 + 8 + 4 + VECTOR_DIMENSION\n\n/** Index GPU records (versioned int8 or legacy float32), storing only int8 vectors in Lucene. */\nfun main(args: Array<String>) {\n    buildSemanticIndexFromVectors(args)\n}\n\ninternal fun buildSemanticIndexFromVectors(args: Array<String>) {\n    require(args.size == 6) {\n        "Usage: BuildSemanticIndexFromVectors <seforim.db> <model-dir> <vectors.bin-or-directory> <output-root> <shard-index> <shard-count>"\n    }\n    val db = Path.of(args[0]).toAbsolutePath()\n    val modelDir = Path.of(args[1]).toAbsolutePath()\n    val vectors = Path.of(args[2]).toAbsolutePath()\n    val root = Path.of(args[3]).toAbsolutePath()\n    val shardIndex = args[4].toInt()\n    val shardCount = args[5].toInt()\n    require(shardCount > 0 && shardIndex in 0 until shardCount)\n    require(Files.isRegularFile(db))\n    val vectorFiles = if (Files.isDirectory(vectors)) {\n        require(shardCount == 1) { "Multiple vector files require a unified index" }\n        Files.list(vectors).use { stream ->\n            stream.filter { Files.isRegularFile(it) && it.fileName.toString().endsWith(".bin") }\n                .sorted()\n                .toList()\n        }\n    } else listOf(vectors)\n    require(vectorFiles.isNotEmpty()) { "No vector files in $vectors" }\n    val inputs = vectorFiles.map { path ->\n        require(Files.isRegularFile(path))\n        val int8 = Files.newInputStream(path).use { input ->\n            input.readNBytes(8).contentEquals(Int8Vectors.FILE_MAGIC.toByteArray(Charsets.US_ASCII))\n        }\n        val headerBytes = if (int8) 12L else 0L\n        val recordBytes = if (int8) INT8_RECORD_BYTES else FLOAT_RECORD_BYTES\n        require(Files.size(path) >= headerBytes && (Files.size(path) - headerBytes) % recordBytes == 0L) {\n            "Truncated vector file: $path"\n        }\n        VectorInput(path, int8, recordBytes, (Files.size(path) - headerBytes) / recordBytes)\n    }\n    val model = modelDir.resolve("seforim-embed-round2-int8.onnx")\n    val tokenizer = modelDir.resolve("tokenizer.json")\n    require(Files.isRegularFile(model) && Files.isRegularFile(tokenizer))\n    val expected = inputs.sumOf { it.count }\n    require(expected > 0) { "Empty vector file: $vectors" }\n    val output = root.resolve("shard-%02d".format(shardIndex))\n    Files.createDirectories(output)\n    Files.deleteIfExists(output.resolve("semantic.properties"))\n    var indexed = 0L\n    FSDirectory.open(output).use { directory ->\n        IndexWriter(directory, IndexWriterConfig(StandardAnalyzer()).apply {\n            openMode = IndexWriterConfig.OpenMode.CREATE\n        }).use { writer ->\n            for ((path, int8, recordBytes, count) in inputs) {\n                val record = ByteArray(recordBytes)\n                DataInputStream(BufferedInputStream(Files.newInputStream(path), 1 shl 20)).use { input ->\n                    if (int8) {\n                        input.skipNBytes(8)\n                        require(Integer.reverseBytes(input.readInt()) == VECTOR_DIMENSION) { "Invalid vector dimension" }\n                    }\n                    repeat(Math.toIntExact(count)) {\n                        input.readFully(record)\n                        val bytes = ByteBuffer.wrap(record).order(ByteOrder.LITTLE_ENDIAN)\n                        val lineId = bytes.long\n                        val bookId = bytes.long\n                        val isBaseBook = bytes.int\n                        require(lineId > 0 && lineId % shardCount == shardIndex.toLong() && bookId > 0)\n                        val vector = if (int8) ByteArray(VECTOR_DIMENSION).also { bytes.get(it) }\n                            else Int8Vectors.quantize(FloatArray(VECTOR_DIMENSION) { bytes.float })\n                        require(vector.any { it != 0.toByte() } && vector.none { it == (-128).toByte() }) {\n                            "Invalid int8 vector for line $lineId"\n                        }\n                        writer.addDocument(Document().apply {\n                            add(StoredField("line_id", lineId))\n                            add(StoredField("book_id", bookId))\n                            add(IntPoint("book_id", bookId.toInt()))\n                            add(IntPoint("is_base_book", isBaseBook))\n                            add(KnnByteVectorField("vec", vector, VectorSimilarityFunction.COSINE))\n                        })\n                        indexed++\n                        if (indexed % 100_000L == 0L) println("shard $shardIndex: indexed $indexed/$expected")\n                    }\n                }\n            }\n            // One HNSW graph for the whole corpus, rather than a KNN traversal per segment.\n            writer.forceMerge(1)\n            writer.commit()\n        }\n    }\n    val properties = Properties().apply {\n        setProperty("format", "zayit-round2-1")\n        setProperty("vectorEncoding", Int8Vectors.ENCODING)\n        setProperty("dimension", VECTOR_DIMENSION.toString())\n        setProperty("shardIndex", shardIndex.toString())\n        setProperty("shardCount", shardCount.toString())\n        setProperty("eligible", indexed.toString())\n        setProperty("indexed", indexed.toString())\n        setProperty("databaseSha256", sha256(db))\n        setProperty("modelSha256", sha256(model))\n        setProperty("tokenizerSha256", sha256(tokenizer))\n    }\n    Files.newOutputStream(output.resolve("semantic.properties")).use { properties.store(it, "Zayit Round 2 GPU index") }\n    println("Completed shard $shardIndex/$shardCount: $indexed vectors")\n}\n\nprivate data class VectorInput(val path: Path, val int8: Boolean, val recordBytes: Int, val count: Long)\n', 'SeforimLibrary/generator/packaging/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/packaging/PackageSemanticBundle.kt': 'package io.github.kdroidfilter.seforimlibrary.packaging\n\nimport com.github.luben.zstd.ZstdOutputStream\nimport io.github.kdroidfilter.seforimlibrary.search.VectorSearcher\nimport org.apache.commons.compress.archivers.tar.TarArchiveEntry\nimport org.apache.commons.compress.archivers.tar.TarArchiveOutputStream\nimport java.io.BufferedOutputStream\nimport java.io.OutputStream\nimport java.nio.file.Files\nimport java.nio.file.Path\nimport java.nio.file.StandardCopyOption\nimport java.security.MessageDigest\n\nprivate const val MODEL_FILE = "seforim-embed-round2-int8.onnx"\nprivate const val MODEL_SHA = "659226865abd3a1bc833565ae6b2e2f48abdd7136285824a12966d4d3294cbf8"\nprivate const val TOKENIZER_SHA = "0664287976ecb078bdfd8f5e5515dc87d8cb7f985a79a481aa1cdf7a7321c0e9"\n\n/** Package a complete local semantic index without including the database itself. */\nfun main(args: Array<String>) {\n    require(args.size in 5..6) {\n        "Usage: PackageSemanticBundle <seforim.db> <model-dir> <index-dir> <output.tar.zst> <split-bytes> [zstd-level]"\n    }\n    val db = Path.of(args[0]).toAbsolutePath()\n    val model = Path.of(args[1]).toAbsolutePath()\n    val index = Path.of(args[2]).toAbsolutePath()\n    val output = Path.of(args[3]).toAbsolutePath()\n    val splitBytes = args[4].toLong()\n    val compressionLevel = args.getOrNull(5)?.toInt() ?: 22\n    require(compressionLevel in 1..22)\n    require(splitBytes > 0 && output.fileName.toString().endsWith(".tar.zst"))\n    require(Files.isRegularFile(db) && Files.isDirectory(index))\n    require(sha256(model.resolve(MODEL_FILE)) == MODEL_SHA) { "Unexpected Round 2 model" }\n    require(sha256(model.resolve("tokenizer.json")) == TOKENIZER_SHA) { "Unexpected Round 2 tokenizer" }\n    VectorSearcher(index, 256, db, model).use { }\n\n    val files = Files.walk(index).use { stream ->\n        stream.filter { Files.isRegularFile(it) }.sorted().toList()\n    }\n    require(files.isNotEmpty()) { "The semantic index is empty" }\n    Files.createDirectories(output.parent)\n    val split = SplitArchiveOutput(output, splitBytes)\n    try {\n        ZstdOutputStream(BufferedOutputStream(split, 1 shl 20), compressionLevel).use { zstd ->\n            TarArchiveOutputStream(zstd).use { tar ->\n                tar.setLongFileMode(TarArchiveOutputStream.LONGFILE_POSIX)\n                tar.setBigNumberMode(TarArchiveOutputStream.BIGNUMBER_POSIX)\n                add(tar, model.resolve(MODEL_FILE), "model/$MODEL_FILE")\n                add(tar, model.resolve("tokenizer.json"), "model/tokenizer.json")\n                files.forEach { file ->\n                    add(tar, file, "index/${index.relativize(file).toString().replace(\'\\\\\', \'/\')}")\n                }\n                tar.finish()\n            }\n        }\n        val archives = split.finish()\n        val manifest = output.resolveSibling("semantic-bundle.json")\n        val databaseSha = sha256(db)\n        Files.writeString(\n            manifest,\n            """{"format":"zayit-round2-1","databaseSha256":"$databaseSha","archiveType":"tar.zst","parts":${archives.size}}""" + "\\n",\n        )\n        println("Semantic bundle ready: ${archives.joinToString()} (database SHA-256 $databaseSha)")\n        println("Manifest: $manifest")\n    } catch (failure: Throwable) {\n        split.abort()\n        throw failure\n    }\n}\n\nprivate fun add(tar: TarArchiveOutputStream, file: Path, name: String) {\n    val entry = TarArchiveEntry(file.toFile(), name)\n    tar.putArchiveEntry(entry)\n    Files.newInputStream(file).use { input -> input.copyTo(tar, 1 shl 20) }\n    tar.closeArchiveEntry()\n}\n\nprivate fun sha256(path: Path): String {\n    val digest = MessageDigest.getInstance("SHA-256")\n    Files.newInputStream(path).use { input ->\n        val buffer = ByteArray(1 shl 20)\n        while (true) {\n            val size = input.read(buffer)\n            if (size < 0) break\n            digest.update(buffer, 0, size)\n        }\n    }\n    return digest.digest().joinToString("") { "%02x".format(it) }\n}\n\n/** One tar.zst file below the threshold; otherwise sequential parts of one compressed stream. */\nprivate class SplitArchiveOutput(private val target: Path, private val maxPartBytes: Long) : OutputStream() {\n    private var part = 1\n    private var partBytes = 0L\n    private var closed = false\n    private var output = BufferedOutputStream(Files.newOutputStream(temporary(part)), 1 shl 20)\n\n    private fun temporary(number: Int): Path = target.resolveSibling("${target.fileName}.building.part%02d".format(number))\n\n    override fun write(value: Int) {\n        val one = byteArrayOf(value.toByte())\n        write(one, 0, 1)\n    }\n\n    override fun write(bytes: ByteArray, offset: Int, length: Int) {\n        check(!closed)\n        var position = offset\n        var remaining = length\n        while (remaining > 0) {\n            if (partBytes == maxPartBytes) {\n                output.close()\n                part++\n                partBytes = 0\n                output = BufferedOutputStream(Files.newOutputStream(temporary(part)), 1 shl 20)\n            }\n            val count = minOf(remaining.toLong(), maxPartBytes - partBytes).toInt()\n            output.write(bytes, position, count)\n            position += count\n            remaining -= count\n            partBytes += count\n        }\n    }\n\n    override fun flush() = output.flush()\n\n    override fun close() {\n        if (!closed) {\n            output.close()\n            closed = true\n        }\n    }\n\n    fun finish(): List<Path> {\n        close()\n        val paths = (1..part).map { number ->\n            if (part == 1) target else target.resolveSibling("${target.fileName}.part%02d".format(number))\n        }\n        require(paths.none { Files.exists(it) }) { "Semantic bundle output already exists" }\n        val moved = mutableListOf<Path>()\n        try {\n            paths.forEachIndexed { index, path ->\n                Files.move(temporary(index + 1), path, StandardCopyOption.ATOMIC_MOVE)\n                moved.add(path)\n            }\n        } catch (failure: Throwable) {\n            moved.forEach { runCatching { Files.deleteIfExists(it) } }\n            throw failure\n        }\n        return paths\n    }\n\n    fun abort() {\n        runCatching { close() }\n        (1..part).forEach { runCatching { Files.deleteIfExists(temporary(it)) } }\n    }\n}\n'}
for name, content in embedded_files.items():
    target = repo / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
print("Installed embedded release tools:", len(embedded_files))


In [ ]:
import sys

DB_RELEASE_TAG = 'REPLACE_WITH_NEW_DATABASE_RELEASE_TAG'
PDF_RELEASE_TAG = ''  # Empty selects the latest PDF release and records its tag/digest.
PUBLISH_TAG = DB_RELEASE_TAG + '-full'
assert not DB_RELEASE_TAG.startswith('REPLACE_'), 'יש להזין את תג המסד החדש'
command = [sys.executable, str(repo / 'scripts/kaggle_semantic_bundle.py'),
           '--repo', str(repo), '--db-release-tag', DB_RELEASE_TAG,
           '--publish-tag', PUBLISH_TAG, '--all-distributions',
           '--work', '/kaggle/temp/zayit-full-' + DB_RELEASE_TAG,
           '--output', '/kaggle/working', '--batch', '128']
if PDF_RELEASE_TAG:
    command += ['--pdf-release-tag', PDF_RELEASE_TAG]
subprocess.run(command, check=True)


In [ ]:
print((Path('/kaggle/working') / 'published-release.txt').read_text())
print((Path('/kaggle/working') / 'distributions.json').read_text())


## לאחר סיום

ההפצות כבר פורסמו בגיטהאב. בפלט המחברת נמצאים distributions.json, checksums.sha256 ו־published-release.txt עם הקישור. הארכיונים עצמם זמינים ב־Release ולא נשמרים בפלט המחברת. במקרה כישלון בדקו את הלוג ואת טיוטות ה־Release; גרסת הבסיס המקורית נשמרת.
